# 3장 01. 컨테이너 기초와 Dockerfile 확인

이 notebook은 Dockerfile과 compose 파일을 보면서 container 실습의 최소 개념을 확인합니다. 여기서는 Kubernetes까지 가지 않습니다. Linux 기반 container가 무엇인지, image와 container가 어떻게 다른지, layered filesystem과 isolation을 아주 가볍게 확인한 뒤 간단한 API container 실행 파일을 읽습니다.


## 기본 개념: container를 너무 어렵게 보지 않기

컨테이너 이미지는 실행에 필요한 파일 묶음입니다. 예를 들어 Python 코드, 설치할 package, 실행 명령, 환경 변수 기본값 같은 것이 이미지 안에 들어갑니다. 이미지는 아직 실행 중인 프로그램이 아니라 실행 준비물입니다.

컨테이너는 그 이미지를 실제로 실행한 상태입니다. 같은 이미지로 컨테이너를 여러 개 띄울 수 있고, 각 컨테이너는 서로 다른 process처럼 동작합니다. 그래서 `image`는 recipe 또는 template에 가깝고, `container`는 그 recipe로 만든 실행 중인 인스턴스에 가깝습니다.

Linux container는 host의 Linux kernel을 공유합니다. OS 전체를 새로 부팅하는 VM과 다르게, namespace, cgroup, mount, overlay filesystem 같은 Linux 기능으로 파일, process, network, resource 사용량을 분리해 보이게 합니다. macOS와 Windows에서 Docker를 쓸 때도 내부에는 Linux VM 또는 WSL 2 backend가 있어 Linux container를 실행합니다.

Layered filesystem은 이미지를 여러 층으로 쌓는 방식입니다. `FROM`으로 시작한 base image 위에 package 설치 layer, source copy layer, 설정 layer가 차례로 쌓입니다. Dockerfile의 각 명령이 대체로 layer를 만들기 때문에, Dockerfile을 읽으면 이미지가 어떤 순서로 만들어지는지 따라갈 수 있습니다.

이번 실습의 목표는 Docker 전문가가 되는 것이 아닙니다. `FROM`, `COPY`, `RUN`, `ENV`, `EXPOSE`, `CMD`가 무슨 역할을 하는지 보고, 간단한 API 서버를 container로 실행할 때 필요한 파일이 무엇인지 확인하는 정도면 충분합니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
from pathlib import Path

file_options = {
    "Dockerfile": [
        Path("demos/ch03_docker_kubernetes/Dockerfile"),
        Path("../../demos/ch03_docker_kubernetes/Dockerfile"),
        Path("artifacts/deployment/chapter_03/Dockerfile"),
        Path("../artifacts/deployment/chapter_03/Dockerfile"),
    ],
    "compose": [Path("compose.yaml"), Path("../../compose.yaml"), Path("artifacts/deployment/chapter_03/compose.yaml"), Path("../artifacts/deployment/chapter_03/compose.yaml")],
    "argocd_app": [Path("demos/ch03_docker_kubernetes/argocd/application.yaml"), Path("../../demos/ch03_docker_kubernetes/argocd/application.yaml"), Path("artifacts/deployment/chapter_03/argocd/application.yaml"), Path("../artifacts/deployment/chapter_03/argocd/application.yaml")],
}

found_files = {}
for name, candidates in file_options.items():
    found_files[name] = next((path for path in candidates if path.exists()), None)

pd.DataFrame([{"file": name, "exists": path is not None, "path": str(path) if path else "missing"} for name, path in found_files.items()])


## 1. Dockerfile 앞부분 보기

Dockerfile은 이미지를 만드는 순서를 적은 파일입니다. 여기서는 전체 Docker 문법을 배우지 않고, base image, 파일 복사, dependency 설치, 환경 변수, port, 실행 명령이 어디에 보이는지만 확인합니다.


In [ ]:
dockerfile = found_files["Dockerfile"]
if dockerfile is None:
    print("Dockerfile not found")
else:
    print("\n".join(dockerfile.read_text().splitlines()[:12]))


## 2. Dockerfile에서 기본 확인 항목 찾기

다음 코드는 Dockerfile 텍스트에서 container 기본 실습에 필요한 키워드를 찾습니다. `present`가 `True`이면 그 항목이 파일 안에 보인다는 뜻입니다. `False`가 나와도 바로 실패는 아니지만, 왜 없는지 설명할 수 있어야 합니다.


In [ ]:
dockerfile_text = dockerfile.read_text() if dockerfile is not None else ""

dockerfile_checks = [
    {
        "concept": "base image",
        "keyword": "FROM",
        "present": "FROM " in dockerfile_text,
        "why_it_matters": "Python/runtime 기준을 고정합니다.",
    },
    {
        "concept": "linux base image",
        "keyword": "python:3.11-slim",
        "present": "python:3.11-slim" in dockerfile_text,
        "why_it_matters": "운영 Linux container 기준으로 실행된다는 점을 확인합니다.",
    },
    {
        "concept": "layered filesystem",
        "keyword": "COPY/RUN",
        "present": "COPY " in dockerfile_text and "RUN " in dockerfile_text,
        "why_it_matters": "어떤 파일과 dependency가 이미지 layer에 들어가는지 봅니다.",
    },
    {
        "concept": "model artifact",
        "keyword": "chapter_02_baseline.pkl",
        "present": "chapter_02_baseline.pkl" in dockerfile_text,
        "why_it_matters": "2장 기준선 모델이 serving image에 연결되는지 봅니다.",
    },
    {
        "concept": "environment",
        "keyword": "ENV MODEL_*",
        "present": "MODEL_VERSION" in dockerfile_text and "MODEL_THRESHOLD" in dockerfile_text,
        "why_it_matters": "평가 기준과 serving 기준을 응답/로그에서 추적합니다.",
    },
    {
        "concept": "network",
        "keyword": "EXPOSE",
        "present": "EXPOSE " in dockerfile_text,
        "why_it_matters": "컨테이너가 어떤 port로 요청을 받을지 확인합니다.",
    },
    {
        "concept": "entrypoint",
        "keyword": "CMD",
        "present": "CMD " in dockerfile_text,
        "why_it_matters": "실제로 어떤 serving process가 시작되는지 확인합니다.",
    },
    {
        "concept": "least privilege",
        "keyword": "USER",
        "present": "USER " in dockerfile_text,
        "why_it_matters": "운영에서는 non-root 실행 여부를 추가로 확인합니다.",
    },
]

pd.DataFrame(dockerfile_checks)


## 3. compose에서 실행 조건 보기

compose 파일은 로컬에서 여러 container를 함께 띄울 때 사용합니다. 이 과정에서는 FastAPI API와 MLflow 같은 주변 서비스를 local container로 묶어 확인할 수 있습니다. 여기서는 port, volume, environment, healthcheck처럼 실행할 때 필요한 조건만 봅니다.


In [ ]:
compose = found_files["compose"]
compose_text = compose.read_text() if compose is not None else ""

runtime_checks = [
    {
        "check": "environment variables",
        "present": "MODEL_VERSION" in compose_text and "MODEL_THRESHOLD" in compose_text,
        "qa_interpretation": "실행 시 모델 버전과 threshold가 어떻게 주입되는지 봅니다.",
    },
    {
        "check": "port mapping",
        "present": "ports:" in compose_text,
        "qa_interpretation": "외부 요청이 API 또는 MLflow로 들어가는 경로를 확인합니다.",
    },
    {
        "check": "volume mount",
        "present": "volumes:" in compose_text,
        "qa_interpretation": "컨테이너 재생성 후에도 남아야 하는 artifact와 로그 경로를 확인합니다.",
    },
    {
        "check": "health check",
        "present": "healthcheck:" in compose_text or "HEALTHCHECK" in dockerfile_text,
        "qa_interpretation": "서비스가 트래픽을 받기 전 준비 상태 기준이 있는지 봅니다.",
    },
    {
        "check": "resource limit",
        "present": "mem_limit" in compose_text or "cpus:" in compose_text or "resources:" in compose_text,
        "qa_interpretation": "latency 급증, OOM kill, 반복 restart 원인을 추적할 단서를 봅니다.",
    },
    {
        "check": "observability handoff",
        "present": "loki" in compose_text.lower() or "grafana" in compose_text.lower(),
        "qa_interpretation": "4장에서 로그와 dashboard로 이어질 실행 경로를 확인합니다.",
    },
]

pd.DataFrame(runtime_checks)


## 4. 실습 해석

Dockerfile에서 확인할 것은 “이 모델 API를 어떤 환경에서 실행할 것인가”입니다. `FROM`은 출발 이미지, `COPY`는 어떤 파일을 이미지에 넣는지, `RUN`은 어떤 dependency를 설치하는지, `CMD`는 container가 시작될 때 무엇을 실행하는지 보여 줍니다.

compose에서 확인할 것은 “로컬에서 어떻게 띄우고 확인할 것인가”입니다. port가 있어야 브라우저나 `curl`로 접근할 수 있고, volume이 있으면 host 파일과 container 파일이 연결됩니다. healthcheck가 있으면 container가 켜졌는지뿐 아니라 서비스가 응답 가능한지도 확인할 수 있습니다.

이 단계에서 Kubernetes, Argo CD, KServe까지 설명하려고 하지 않습니다. 먼저 image와 container 차이를 잡고, Dockerfile로 아주 간단한 container를 만들 수 있다는 감각을 만든 뒤 다음 단계에서 MLflow container와 FastAPI/Compose로 넘어갑니다.
